In [ ]:
inputs = [35,25]

In [ ]:
type(inputs)

In [ ]:
weights = [0.8, 0.1]

# **Sum Function**

In [ ]:
def sum_func(inputs: list, weights: list):
    # res = 0
    # for _input, _weight in zip(inputs, weights):
    #     res += _input * _weight
    return sum(input_ * weight_ for input_, weight_ in zip(inputs, weights))

In [ ]:
sum_func(inputs, weights)

# **Step Function**

In [ ]:
def step_func(sum):
    return int(sum >= 1)

In [ ]:
s = sum_func(inputs, weights)

In [ ]:
step_func(s)

# **Using numpy to compute sum effectively**

In [ ]:
import numpy as np

In [ ]:
def sum_func(inputs, weights):
    return np.array(inputs) @ np.array(weights)

In [ ]:
sum_func(inputs, weights)

## **Gradient Descent: First Attempt at Implementation**

In [4]:
import numpy as np

def grad(features, rows, learning_rate):
    """
    assuming features = no. of features,
    rows is in the shape [[[x1, x2, x3, ...], y], [...], [...]],
    and learning rate is... learning rate :)
    """
    weights = np.random.randn(features)
    bias = 0

    for row in rows:
        x, y = row

        prediction = weights @ x + bias
        error = prediction - y

        djdw = (error) * (x) * 2.0
        djdb = (error) * 2.0

        weights -= djdw * learning_rate
        bias -= djdb * learning_rate
    return weights, bias

In [5]:
# testing with f(x) = 3x + 5
def target_func(x):
    return 3 * x + 5

In [6]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [7]:
print(grad(1, data, 0.1))

(array([-8.99734808e+241]), np.float64(-9.08818308046604e+239))


## **Bad results because no epochs and high learning rate**

In [8]:
import numpy as np

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias
            error = prediction - y

            total_loss += error**2

            djdw = 2 * error * x
            djdb = 2 * error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "MSE:", total_loss / len(rows))

    return weights, bias

In [ ]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [ ]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [ ]:
print(grad(1, data, 0.0001))

## **Attempting to apply it to a real dataset**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [ ]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [ ]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [ ]:
x_train = transformations.fit_transform(x_train)

In [ ]:
x_train

In [ ]:
x_test = transformations.transform(x_test)

In [ ]:
train_data = list(zip(x_train, y_train))

In [ ]:
test_data = zip(x_test, y_test)

In [ ]:
run = False # this takes way too long, adviced not to run.
if run:
    weights, bias = grad(x_train.shape[1], train_data, learning_rate = 1e-6, epochs = 1000)

In [ ]:
if run: 
    predictions = x_test @ weights + bias
    predictions.shape

In [ ]:
if run:
    mse = sum(((y_test - predictions) ** 2)) / x_test.shape[0]
    mse

# **Optimizing for larger datasets**

In [ ]:
import numpy as np

def grad(features, x, y, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    x, y = x.astype(np.float32), y.astype(np.float32)
    bias = 0.0

    for epoch in range(epochs):

        prediction = x @ weights + bias
        error = prediction - y

        djdw = (2 / len(x)) * (x.T @ error)
        djdb = (2 / len(x)) * np.sum(error)

        weights -= learning_rate * djdw
        bias -= learning_rate * djdb
        if epoch % 50 == 0:
            print(epoch, np.mean(error**2))
    return weights, bias

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [ ]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [ ]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [ ]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1,1)
).flatten()

In [ ]:
x_train = transformations.fit_transform(x_train)

In [ ]:
x_test = transformations.transform(x_test)

In [ ]:
weights, bias = grad(x_train.shape[1], x_train, y_train_scaled, learning_rate = 0.008, epochs = 1000)

In [ ]:
predictions = x_test @ weights + bias   

In [ ]:
predictions = y_scaler.inverse_transform(
    predictions.reshape(-1,1)
).flatten()

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mean_absolute_error(y_test, predictions)

In [ ]:
r2_score(y_test, predictions)

# **Classification Model - Logistic Regression**

In [ ]:
import numpy as np

# stochastic approach

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    sigmoid = lambda x: 1 / (1 + np.e**(-x))

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.
            error = prediction - y

            total_loss += -(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = error * x
            djdb = error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss / len(rows))

    return weights, bias

In [ ]:
def predict(x, weights, bias, threshold):
    sigmoid = lambda x: 1 / (1 + np.e**(-x))
    return 1 if sigmoid(weights @ x + bias) > threshold else 0

In [ ]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

data = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    y = int(x1 + x2 >= 10)

    data.append((np.array([x1, x2]), y))

In [ ]:
train_size = int(0.8 * 500)

train_data = data[:train_size]
test_data = data[train_size:]

In [ ]:
weights, bias = grad(2, train_data, 0.1, 1000)

In [ ]:
x_test, y_test = [row[0] for row in test_data], np.array([row[1] for row in test_data])

In [ ]:
y_pred = np.array([predict(x, weights, bias, 0.5) for x in x_test])

In [ ]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

In [ ]:
weights, bias

In [ ]:
import numpy as np  

# batch approach

def grad(x, y, learning_rate, epochs=1000):

    features = x.shape[1]

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    x = x.astype(np.float32)
    
    sigmoid = lambda x: 1 / (1 + np.exp(-x))

    for epoch in range(epochs):

            prediction = x @ weights + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.

            error = prediction - y

            total_loss = -np.mean(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = (x.T @ error) / x.shape[0]
            djdb = np.mean(error)

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

            if epoch % 100 == 0:
                        print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss)

    return weights, bias

In [ ]:
def predict(x, weights, bias, threshold = 0.5):
    sigmoid = lambda x: 1 / (1 + np.exp(-x))
    return np.array([sigmoid(x @ weights + bias) >= threshold])

In [ ]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

x = []
y = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    label = int(x1 + x2 >= 10)

    x.append([x1, x2])
    y.append(label)

x = np.array(x)
y = np.array(y)

In [ ]:
x.shape, y.shape

In [ ]:
train_size = int(0.8 * 500)

x_train, y_train, x_test, y_test = x[:train_size], y[:train_size], x[train_size:], y[train_size:]

In [ ]:
for arr in [x_train, y_train, x_test, y_test]:
    print(arr.shape)

In [ ]:
weights, bias = grad(x_train, y_train, 0.01, 2000)

In [ ]:
y_pred = predict(x_test, weights, bias)

In [ ]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

## **Attempting to implement a neural network**

In [ ]:
class neuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias
        
    def output(self, x):
        return self.sigmoid(x @ self.weights + self.bias)

    def sigmoid(self, x):   
        return 1 / (1 + np.exp(-x))
    

In [ ]:
class network:

    def __init__(self, layer_size, x, y, learning_rate = 0.01, epochs = 1000, threshold = 0.5, verbose = 0):
        """
        naming convention
    
            d()d() => partial diff
            ()f => final neuron related
            ()h => hidden neuron related
            j => cost func(cross entropy loss)
            p => sigmoid
            z => linear form(regression equation before segmoid)
            w => weights
            b => bias

        shapes
            x -> (samples, features)
            y -> (samples,)
            hidden_neurons -> (layer_size,)
            hidden_outputs -> (samples, layer_size)
            prediction -> (predictions,) or (samples,)
            error -> (errors,) or (samples,)
            weights -> (features,)
            final neuron weights -> (layer_size,)
            delta_hidden -> (samples, layer_size)
            
            """  
        features = x.shape[1]
        self.threshold = threshold
        self.hidden_neurons = list([neuron(np.random.randn(features) * 1.0, 0.0) for _ in range(layer_size)])
        self.final_neuron = neuron(np.random.randn(len(self.hidden_neurons)) * 1.0, 0.0)

        for epoch in range(epochs):
            hidden_outputs = self.hidden_outputs(x)
            prediction = self.final_neuron.output(hidden_outputs)
            error = prediction - y

            djdwf = (hidden_outputs.T @ error) / hidden_outputs.shape[0] # loss function wrt final neuron weights
            djdbf = np.mean(error) # loss function wrt final neurons bias

            delta_hidden = ((hidden_outputs * (1 - hidden_outputs)) * (self.final_neuron.weights * error[:, None])) 
            # loss function wrt hidden neuron outputs ^^

            djdwh = (x.T @ delta_hidden) / x.shape[0] # loss function wrt hidden neuron weights
            djdbh = np.mean(delta_hidden, axis = 0) # loss function wrt hidden neuron bias

            for n in range(len(self.hidden_neurons)):
                 self.hidden_neurons[n].weights -= djdwh[:, n] * learning_rate # since shape is (layer_size, 1)
                 self.hidden_neurons[n].bias -= djdbh[n] * learning_rate

            self.final_neuron.weights -= djdwf * learning_rate
            self.final_neuron.bias -= djdbf * learning_rate
            
            prediction = np.clip(
                prediction,
                1e-15,
                1-1e-15
            )

            loss = np.mean(
                -(y*np.log(prediction)
                +(1-y)*np.log(1-prediction))
            )

            if epoch % 1000 == 0:
                if verbose == 1:
                    print(epoch, loss)
                if verbose == 2:
                     print(epoch, loss)
                     print(np.max(np.abs(djdwf)))
                     print(np.max(np.abs(djdwh)))                 
            
    def hidden_outputs(self, x):
            outputs = np.array(list([n.output(x) for n in self.hidden_neurons]))
            return outputs.T

    def predict_proba(self, x):
        hidden_pred = self.hidden_outputs(x)
        prediction = self.final_neuron.output(hidden_pred)
        return prediction
    
    def predict(self, x):
        return (self.predict_proba(x) >= self.threshold).astype(np.int32)

In [ ]:
x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    0,
    1,
    1,
    0
], dtype=np.float32)

In [ ]:
net = network(
    layer_size=4,
    x=x,
    y=y,
    learning_rate=0.1,
    epochs=10000
)

In [ ]:
print(net.predict(x))

## **This is a single-layer neural network for binary classification**

In [ ]:
# matrix style implementation; no neuron class

import numpy as np

class network:

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def __init__(self, layer_size, x, y, learning_rate = 0.01, epochs = 1000, intialization_strength = 0.01, threshold = 0.5, verbose = False):
        """
        naming convention
    
            d()d() => partial diff
            ()f => final neuron related
            ()h => hidden neuron related
            j => cost func(cross entropy loss)
            p => sigmoid
            z => linear form(regression equation before segmoid)
            w => weights
            b => bias

        shapes

            x => (samples, features)
            y => (samples,)
            hidden_weights => (features, layer_size)
            final_weights => (layer_size,)
            prediction => (samples,)
            hidden_outputs => (samples, layer_size)
            delta_hidden => (samples, layer_size)
            djdwh => (features, layer_size)
            """  
        
        self.threshold = threshold
        self.hidden_weights = np.random.randn(x.shape[1], layer_size) * intialization_strength
        self.hidden_biases = np.random.randn(layer_size) * intialization_strength

        self.final_weights = np.random.randn(layer_size) * intialization_strength
        self.final_bias = np.random.randn() * intialization_strength

        for epoch in range(epochs):
            hidden_outputs = self.sigmoid(x @ self.hidden_weights + self.hidden_biases)
            prediction = self.sigmoid(hidden_outputs @ self.final_weights + self.final_bias)
            prediction = np.clip(
                                prediction,
                                1e-15,
                                1-1e-15
                    )
            error = prediction - y
            m = x.shape[0]

            djdwf = hidden_outputs.T @ error / m
            djdbf = np.mean(error)

            delta_hidden = (hidden_outputs * (1 - hidden_outputs)) * (self.final_weights * error[:, None])

            djdwh = (x.T @ delta_hidden) / m
            djdbh = np.mean(delta_hidden, axis= 0)

            self.hidden_weights -= learning_rate * djdwh
            self.hidden_biases -= learning_rate * djdbh


            self.final_weights -= learning_rate * djdwf
            self.final_bias -= learning_rate * djdbf

            if verbose and epoch % 1000 == 0:
                loss = np.mean(
                                -(y*np.log(prediction)
                                +(1-y)*np.log(1-prediction))
                            )
                print(f"Epoch no. {epoch}; Cross-Entropy Loss: {loss}")

    def predict_proba(self, x):
        hidden_outputs = self.sigmoid(x @ self.hidden_weights + self.hidden_biases)
        prediction = self.sigmoid(hidden_outputs @ self.final_weights + self.final_bias)
        prediction = np.clip(
                            prediction,
                            1e-15,
                            1-1e-15
                            )
        return prediction

    def predict(self, x):
        probas = self.predict_proba(x)
        return (probas >= self.threshold).astype(np.int32)

In [ ]:
x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    0,
    1,
    1,
    0
], dtype=np.float32)

In [ ]:
net = network(
    layer_size=4,
    x=x,
    y=y,
    learning_rate=1,
    intialization_strength=1,
    epochs=10000,
    verbose = 2
)

In [ ]:
print(list(format(y, ".25f") for y in net.predict_proba(x)))